# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/menna890/-Explaining-Search-Performance-Gaps-Using-Ranking-Signals/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*A page is worth reviewing if it ranks in a visible position (top 5) but shows multiple decay signals — old content, stale updates, thin text, or no backlinks. The more decay signals it has while being visible, the higher the priority.*

In [2]:
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules

REPO_URL = "https://github.com/menna890/flyrank-ml-internship-Explaining-Search-Performance"
REPO_DIR = "flyrank-search-performance"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
            check=True
        )

    os.chdir(REPO_DIR)

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
        check=True
    )

else:
    while not os.path.isdir("data") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())

assert os.path.exists(
    "data/ml_features_march_2026.parquet"
), "Dataset not found — are you at the repo root?"

print("Dataset found. You're ready.")


Working dir: /content/flyrank-search-performance
Dataset found. You're ready.


In [3]:
import pandas as pd
import numpy as np

df_raw = pd.read_parquet('data/ml_features_march_2026.parquet')
df = df_raw.copy()

# Display the first few rows of the DataFrame
print("DataFrame loaded successfully. Here's the head:")
print(df.head())
print(df.columns)


DataFrame loaded successfully. Here's the head:
            client_hash_id           content_hash_id  avg_position  \
0  client_62f4a7e64f5e0096  content_2e6360ad20fd7107      5.908100   
1  client_62f4a7e64f5e0096  content_ac8663da7484669a      6.419872   
2  client_62f4a7e64f5e0096  content_d49a012dcb924e31      5.177774   
3  client_62f4a7e64f5e0096  content_614baf2af4330bd7      4.685335   
4  client_62f4a7e64f5e0096  content_4a1ca0fa5c177e0c      5.333333   

   total_clicks  total_impressions       ctr  days_with_data  ga4_sessions  \
0           1.0              884.0  0.001131              27           0.0   
1           0.0               28.0  0.000000              13           0.0   
2           0.0              329.0  0.000000              31           0.0   
3           1.0              772.0  0.001295              31           0.0   
4           0.0               12.0  0.000000               8           0.0   

   engaged_sessions  scroll_events  ...  has_keyword_data  has

In [4]:

import pandas as pd
import numpy as np


# ── 2. Base rate
base_rate = df['is_underperforming'].mean()
print(f"Base rate (is_underperforming): {base_rate:.3f}  ({base_rate*100:.1f}%)")
print(f"Total pages: {len(df):,}\n")

# ── 3. Check distributions of the features we'll use ──
rule_features = ['avg_position', 'content_age_days', 'days_since_update',
                 'word_count', 'backlinks']
print("Feature distributions (for rule design):")
print(df[rule_features].describe().round(1))

# ── 4. How many pages meet each condition? ──
print("\n--- Condition coverage ---")
print(f"avg_position <= 5 (visible):      {(df['avg_position'] <= 5).mean()*100:.1f}%")
print(f"content_age_days >= 365 (old):    {(df['content_age_days'] >= 365).mean()*100:.1f}%")
print(f"days_since_update >= 180 (stale): {(df['days_since_update'] >= 180).mean()*100:.1f}%")
print(f"0 < word_count < 500 (thin):      {((df['word_count'] > 0) & (df['word_count'] < 500)).mean()*100:.1f}%")
print(f"backlinks == 0 (no authority):    {(df['backlinks'] == 0).mean()*100:.1f}%")

Base rate (is_underperforming): 0.703  (70.3%)
Total pages: 175,304

Feature distributions (for rule design):
       avg_position  content_age_days  days_since_update  word_count  \
count      175304.0          175304.0           175304.0    175304.0   
mean           17.1             184.7              -47.1      1880.9   
std            18.3             123.8               44.0      1601.5   
min             0.1               0.0              -97.0         0.0   
25%             5.5              69.0              -78.0         0.0   
50%             9.0             191.0              -50.0      2345.0   
75%            22.0             260.0              -48.0      2928.0   
max           309.0             494.0              303.0     29341.0   

       backlinks  
count   175304.0  
mean       151.0  
std       6169.1  
min          0.0  
25%          0.0  
50%          0.0  
75%          0.0  
max     854412.0  

--- Condition coverage ---
avg_position <= 5 (visible):      21.5%
co

## 2. Build the Ranked Queue

### Rule (plain words)
A page gets a baseline score only if it is **visible** (`avg_position &lt;= 5`).  
The score equals how many **decay signals** it carries:
- **old**: `content_age_days &gt;= 200`
- **stale**: `ever_optimized == 0` (never optimized)
- **thin**: `has_word_count == 0` (no length data)
- **no_backlinks**: `backlinks == 0`

**Formula:** `score = visible × (old + stale + thin + no_backlinks)`

**Reason codes** concatenate every signal that fired, e.g. `old+stale+no_backlinks`.

### Why this rule?
Pages that already rank well but show multiple decay signals are the fastest wins:  
fix the decay → recover clicks immediately. Pages not in top 5 are deprioritized because their gap may be ranking, not content quality.

In [5]:
import pandas as pd
import numpy as np
import os


# ── 2. Build binary condition flags ──
df['visible']        = (df['avg_position'] <= 5).astype(int)
df['old']            = (df['content_age_days'] >= 200).astype(int)
df['stale']          = (df['ever_optimized'] == 0).astype(int)
df['thin']           = (df['has_word_count'] == 0).astype(int)
df['no_backlinks']   = (df['backlinks'] == 0).astype(int)

# ── 3. Decay signal count & baseline score ──
df['decay_signals']   = df['old'] + df['stale'] + df['thin'] + df['no_backlinks']
df['baseline_score']  = df['visible'] * df['decay_signals']

# ── 4. Reason codes ──
def build_reason(row):
    if row['visible'] == 0:
        return 'not_visible'
    parts = []
    if row['old'] == 1:          parts.append('old')
    if row['stale'] == 1:        parts.append('stale')
    if row['thin'] == 1:         parts.append('thin')
    if row['no_backlinks'] == 1: parts.append('no_backlinks')
    return '+'.join(parts) if parts else 'visible_ok'

df['reason_code'] = df.apply(build_reason, axis=1)

# ── 5. Rank by score (descending) ──
df = df.sort_values('baseline_score', ascending=False).reset_index(drop=True)
df['rank'] = np.arange(1, len(df) + 1)

# ── 6. Precision@K evaluation ──
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

print("=" * 50)
print("BASELINE EVALUATION")
print("=" * 50)
print(f"Base rate (random):           {df['is_underperforming'].mean():.3f}")

for k in [20, 50, 100, 500]:
    p = precision_at_k(df['baseline_score'].values,
                       df['is_underperforming'].values, k)
    print(f"Precision@{k:3d}:                 {p:.3f}")

# ── 7. Score distribution ──
print("\n--- Score distribution ---")
print(df['baseline_score'].value_counts().sort_index())

# ── 8. Write CSV ──
os.makedirs('work/outputs', exist_ok=True)

out_cols = [
    'content_hash_id', 'client_hash_id', 'rank', 'baseline_score',
    'reason_code', 'decay_signals', 'visible',
    'avg_position', 'content_age_days', 'ever_optimized',
    'has_word_count', 'backlinks',
    'total_clicks', 'total_impressions', 'is_underperforming'
]

df[out_cols].to_csv('work/outputs/baseline_action_score.csv', index=False)
print(f"\n Saved: work/outputs/baseline_action_score.csv  ({len(df):,} rows)")

BASELINE EVALUATION
Base rate (random):           0.703
Precision@ 20:                 0.500
Precision@ 50:                 0.540
Precision@100:                 0.470
Precision@500:                 0.476

--- Score distribution ---
baseline_score
0    139119
1      6674
2     15398
3      8562
4      5551
Name: count, dtype: int64

 Saved: work/outputs/baseline_action_score.csv  (175,304 rows)


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [6]:
# ── Load the ranked output ──
df_ranked = pd.read_csv('work/outputs/baseline_action_score.csv')

# ── Show top 20 with full context ──
top20 = df_ranked.head(20).copy()

# Add action & confidence columns
def get_action(row):
    actions = []
    if 'old' in row['reason_code']:
        actions.append("Audit content freshness")
    if 'stale' in row['reason_code']:
        actions.append("Schedule optimization/rewrite")
    if 'thin' in row['reason_code']:
        actions.append("Expand word count & depth")
    if 'no_backlinks' in row['reason_code']:
        actions.append("Build authority links")
    return " + ".join(actions) if actions else "Review position only"

def get_confidence(row):
    if row['decay_signals'] >= 3:
        return "High"
    elif row['decay_signals'] == 2:
        return "Medium"
    else:
        return "Low"

def get_what_wrong(row):
    if row['is_underperforming'] == 0:
        return "Page is actually fine — false positive"
    if row['total_impressions'] < 100:
        return "Too little traffic to matter — noise"
    if 'thin' in row['reason_code'] and row['has_word_count'] == 0:
        return "Missing word_count may be data gap, not thin content"
    return "Page needs technical fix, not content refresh"

top20['action'] = top20.apply(get_action, axis=1)
top20['confidence'] = top20.apply(get_confidence, axis=1)
top20['what_would_make_wrong'] = top20.apply(get_what_wrong, axis=1)

# Display
display_cols = ['rank', 'baseline_score', 'reason_code', 'avg_position',
                'content_age_days', 'ever_optimized', 'has_word_count',
                'backlinks', 'total_clicks', 'total_impressions',
                'is_underperforming', 'action', 'confidence', 'what_would_make_wrong']

print("=" * 80)
print("TOP 20 BASELINE REVIEW")
print("=" * 80)
for i, row in top20.iterrows():
    print(f"\n--- Rank {row['rank']} | Score: {row['baseline_score']} | Reason: {row['reason_code']} ---")
    print(f"  Position: {row['avg_position']:.1f} | Age: {row['content_age_days']:.0f}d | "
          f"Optimized: {'Yes' if row['ever_optimized'] else 'No'} | "
          f"WordCount: {'Yes' if row['has_word_count'] else 'No'} | "
          f"Backlinks: {row['backlinks']:.0f}")
    print(f"  Clicks: {row['total_clicks']:.0f} | Impressions: {row['total_impressions']:.0f} | "
          f"Underperforming: {'YES' if row['is_underperforming'] else 'NO'}")
    print(f"  → Action: {row['action']}")
    print(f"  → Confidence: {row['confidence']}")
    print(f"  → What would make it wrong: {row['what_would_make_wrong']}")

# Summary stats for top 20
print("\n" + "=" * 80)
print("TOP 20 SUMMARY")
print("=" * 80)
print(f"True underperforming: {top20['is_underperforming'].sum()}/20 ({top20['is_underperforming'].mean()*100:.0f}%)")
print(f"Average impressions:  {top20['total_impressions'].mean():,.0f}")
print(f"Average clicks:       {top20['total_clicks'].mean():,.0f}")
print(f"Reason codes:")
print(top20['reason_code'].value_counts())

TOP 20 BASELINE REVIEW

--- Rank 1 | Score: 4 | Reason: old+stale+thin+no_backlinks ---
  Position: 2.2 | Age: 260d | Optimized: No | WordCount: No | Backlinks: 0
  Clicks: 30 | Impressions: 5496 | Underperforming: NO
  → Action: Audit content freshness + Schedule optimization/rewrite + Expand word count & depth + Build authority links
  → Confidence: High
  → What would make it wrong: Page is actually fine — false positive

--- Rank 2 | Score: 4 | Reason: old+stale+thin+no_backlinks ---
  Position: 2.9 | Age: 260d | Optimized: No | WordCount: No | Backlinks: 0
  Clicks: 0 | Impressions: 1183 | Underperforming: YES
  → Action: Audit content freshness + Schedule optimization/rewrite + Expand word count & depth + Build authority links
  → Confidence: High
  → What would make it wrong: Missing word_count may be data gap, not thin content

--- Rank 3 | Score: 4 | Reason: old+stale+thin+no_backlinks ---
  Position: 3.7 | Age: 271d | Optimized: No | WordCount: No | Backlinks: 0
  Clicks: 0 |

## 4. Weak Picks & Leakage Check

### Weak picks in the top 20
I expect some false positives because the rule is blunt:
- Pages with `has_word_count = 0` may be comparison articles (not keyword-targeted), so "thin" is misleading.
- Pages with zero backlinks may still rank well via brand authority.
- A page with `is_underperforming = 0` in the top 20 is a confirmed false positive.

### Leakage check
I confirm the baseline score uses **only** features knowable before prediction:
- `avg_position`, `content_age_days`, `ever_optimized`, `has_word_count`, `backlinks`
- **No label fields used**: `total_clicks`, `total_impressions`, `ctr`, `expected_clicks`, `underperformance_score`, `is_underperforming` were NOT in the score formula.
- **No product flags**: `optimization_eligible_date` was excluded in ML-04.
- **No future window**: all features come from Jan–Mar 2026; the label is from the same window (cross-sectional snapshot).

The baseline is honest and beatable.

In [7]:
# ── Weak picks analysis ──
print("=" * 60)
print("WEAK PICKS ANALYSIS")
print("=" * 60)

# False positives in top 20
fp_top20 = top20[top20['is_underperforming'] == 0]
print(f"\nFalse positives in top 20: {len(fp_top20)}/20")
if len(fp_top20) > 0:
    print("Ranks:", fp_top20['rank'].tolist())
    print("Reasons:", fp_top20['reason_code'].tolist())

# Low-traffic noise in top 20
low_imp = top20[top20['total_impressions'] < 100]
print(f"\nLow-traffic pages (<100 imp) in top 20: {len(low_imp)}/20")

# Score=4 pages that are NOT underperforming
score4_not_under = df_ranked[(df_ranked['baseline_score'] == 4) &
                              (df_ranked['is_underperforming'] == 0)]
print(f"\nScore=4 pages that are NOT underperforming: {len(score4_not_under):,} "
      f"({len(score4_not_under)/df_ranked[df_ranked['baseline_score']==4].shape[0]*100:.1f}% of all score=4)")

# ── Leakage verification ──
print("\n" + "=" * 60)
print("LEAKAGE CHECK")
print("=" * 60)

score_formula_features = ['visible', 'old', 'stale', 'thin', 'no_backlinks']
label_fields = ['total_clicks', 'total_impressions', 'ctr',
                'expected_clicks', 'underperformance_score', 'is_underperforming']

print("Features USED in score formula:")
for f in score_formula_features:
    print(f"   {f}")

print("\nLabel fields NOT used in score formula:")
for f in label_fields:
    print(f"   {f}")

# Confirm no overlap
overlap = set(score_formula_features) & set(label_fields)
assert len(overlap) == 0, f"LEAKAGE DETECTED: {overlap}"
print("\n NO LEAKAGE: Score formula uses only pre-prediction features.")

WEAK PICKS ANALYSIS

False positives in top 20: 13/20
Ranks: [1, 4, 5, 7, 8, 9, 11, 13, 14, 15, 18, 19, 20]
Reasons: ['old+stale+thin+no_backlinks', 'old+stale+thin+no_backlinks', 'old+stale+thin+no_backlinks', 'old+stale+thin+no_backlinks', 'old+stale+thin+no_backlinks', 'old+stale+thin+no_backlinks', 'old+stale+thin+no_backlinks', 'old+stale+thin+no_backlinks', 'old+stale+thin+no_backlinks', 'old+stale+thin+no_backlinks', 'old+stale+thin+no_backlinks', 'old+stale+thin+no_backlinks', 'old+stale+thin+no_backlinks']

Low-traffic pages (<100 imp) in top 20: 3/20

Score=4 pages that are NOT underperforming: 2,800 (50.4% of all score=4)

LEAKAGE CHECK
Features USED in score formula:
   visible
   old
   stale
   thin
   no_backlinks

Label fields NOT used in score formula:
   total_clicks
   total_impressions
   ctr
   expected_clicks
   underperformance_score
   is_underperforming

 NO LEAKAGE: Score formula uses only pre-prediction features.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.